# 01 — Final 10-Seed Benchmark

Portable final benchmark for all four datasets. All paths are relative to the repository root. Every stochastic benchmark model is refit with the same ten fixed seeds; deterministic references are evaluated once. Model selection uses Train/Validation only and Test is never used to change settings.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, platform
from datetime import datetime, timezone

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == 'notebooks':
    ROOT = ROOT.parent
SRC_DIR = ROOT / 'src'
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'outputs'
CONFIG_PATH = ROOT / 'configs' / 'final_protocol.json'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ.setdefault('PYTHONHASHSEED', '42')
for key in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ.setdefault(key, '1')

protocol = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
FINAL_SEEDS = tuple(int(x) for x in protocol['final_seeds'])
print('Repository root:', ROOT)
print('Final seeds:', FINAL_SEEDS)


## Load the audited implementation and build the final protocol

In [ ]:
import pandas as pd
import torch
import psutil, cpuinfo
from IPython.display import display

import lash_revision_core as core
from lash_revision_core import ExperimentConfig
from lash_hardware_optimized import apply_hardware_patch, probe_gpu_boosters
from lash_per_dataset_hpo import PerDatasetHPOPolicy, protocol_manifest, search_space_manifest, run_per_dataset_hpo_benchmark

apply_hardware_patch()
booster_probe = probe_gpu_boosters()
physical_cores = psutil.cpu_count(logical=False) or 1
logical_threads = psutil.cpu_count(logical=True) or physical_cores
TREE_HORIZON_JOBS = min(8, max(1, physical_cores))
torch.set_num_threads(min(8, max(1, logical_threads)))
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

policy = PerDatasetHPOPolicy(
    computational_budget_hours=48.0,
    bootstrap_reps=int(protocol['bootstrap_replicates']),
    tree_final_seeds=FINAL_SEEDS,
    neural_comparator_final_seeds=FINAL_SEEDS,
    lash_final_seeds=FINAL_SEEDS,
    mechanism_seeds=FINAL_SEEDS,
)

config = ExperimentConfig(
    data_root=DATA_DIR,
    output_root=OUTPUT_DIR,
    dataset_keys=('CLUSTER_1','CLUSTER_2','BDG_EDU','BDG_DORM'),
    run_profile='paper',
    weather_mode='historical_only',
    hpo_seeds=policy.hpo_seeds,
    final_refit_seeds=FINAL_SEEDS,
    primary_hpo_repeats=2,
    external_hpo_repeats=2,
    hpo_trials_per_dimension=1,
    hpo_min_trials=3,
    hpo_max_trials=5,
    max_epochs=policy.max_epochs,
    early_stopping_patience=policy.patience,
    tree_horizon_jobs=TREE_HORIZON_JOBS,
    tree_threads_per_model=1,
    require_cuda=False,
    use_amp=True,
    save_models=True,
    resume=True,
)
display(protocol_manifest(policy, config.dataset_keys))


## Verify the four harmonized inputs and freeze the run contract

In [ ]:
EXPECTED = ('Cluster_1_Harmonized.csv','Cluster_2_Harmonized.csv','BDG_Edu_Harmonized.csv','BDG_Dorm_Harmonized.csv')
rows = []
for filename in EXPECTED:
    path = DATA_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    required = ['Date','Holi','Temp','Humi','WS','Consumption']
    if list(df.columns) != required:
        raise ValueError(f'{filename}: expected {required}, got {list(df.columns)}')
    rows.append({'file': filename, 'rows': len(df), 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})

contract = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'protocol': 'lash_final_10seed_portable_v1',
    'final_seeds': list(FINAL_SEEDS),
    'hpo_seeds': list(policy.hpo_seeds),
    'bootstrap_replicates': int(protocol['bootstrap_replicates']),
    'dependence_block_hours': protocol['dependence_block_hours'],
    'primary_block_hours': int(protocol['primary_block_hours']),
    'practical_relative_nmae_threshold_pct': float(protocol['practical_relative_nmae_threshold_pct']),
    'test_used_for_selection': False,
    'inputs': rows,
}
(OUTPUT_DIR / 'final_run_contract.json').write_text(json.dumps(contract, indent=2), encoding='utf-8')
display(pd.DataFrame(rows))


## Run the complete four-dataset benchmark

In [ ]:
RESULTS = run_per_dataset_hpo_benchmark(config, policy)
print('Completed datasets:', list(RESULTS))


## Completion audit

In [ ]:
summary_path = OUTPUT_DIR / '01_all_datasets_local_hpo_summary.xlsx'
print('Summary:', summary_path)
print('All stochastic benchmark families must contain exactly 10 final refits; deterministic references are evaluated once.')
if summary_path.exists():
    print('Benchmark summary workbook exists: PASS')
else:
    raise RuntimeError('Benchmark summary workbook was not created.')
